In [1]:
import joblib
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("artifacts/risk_model.keras")

# Load preprocessing objects
artifacts = joblib.load("artifacts/preprocessors.joblib")

preprocessor = artifacts["preprocessor"]
label_encoder = artifacts["label_encoder"]
feature_columns = artifacts["feature_columns"]

print("✅ Model loaded successfully!")
print("✅ Preprocessor loaded successfully!")

✅ Model loaded successfully!
✅ Preprocessor loaded successfully!


In [2]:
import pandas as pd
import numpy as np

def predict_risk(accident_data):
    """
    Predict accident risk level from a dictionary of accident details.
    """

    # Convert dictionary to DataFrame
    input_df = pd.DataFrame([accident_data])

    # Ensure column order matches training
    input_df = input_df[feature_columns]

    # Apply preprocessing
    X = preprocessor.transform(input_df)

    # Predict probabilities
    prediction = model.predict(X, verbose=0)

    # Highest probability class
    predicted_index = np.argmax(prediction)

    # Convert number back to label
    predicted_label = label_encoder.inverse_transform([predicted_index])[0]

    # Confidence
    confidence = float(np.max(prediction))

    return predicted_label, confidence

In [3]:
# Sample accident record

sample_accident = {
    "Day_of_week": "Monday",
    "Area_accident_occured": "Residential areas",
    "Lanes_or_Medians": "Two-way (divided with broken lines road marking)",
    "Types_of_Junction": "No junction",
    "Road_surface_conditions": "Dry",
    "Light_conditions": "Daylight",
    "Weather_conditions": "Normal",
    "Time_Period": "Morning",
    "Casualty_class": "Driver or rider"
}

risk, confidence = predict_risk(sample_accident)

print("Predicted Risk Level :", risk)
print("Confidence :", round(confidence * 100, 2), "%")

Predicted Risk Level : Low
Confidence : 47.89 %


In [4]:
test_cases = [
    {
        "Day_of_week": "Monday",
        "Area_accident_occured": "Residential areas",
        "Lanes_or_Medians": "Two-way (divided with broken lines road marking)",
        "Types_of_Junction": "No junction",
        "Road_surface_conditions": "Dry",
        "Light_conditions": "Daylight",
        "Weather_conditions": "Normal",
        "Time_Period": "Morning",
        "Casualty_class": "Driver or rider"
    },

    {
        "Day_of_week": "Saturday",
        "Area_accident_occured": "Office areas",
        "Lanes_or_Medians": "Undivided Two way",
        "Types_of_Junction": "Y Shape",
        "Road_surface_conditions": "Wet or damp",
        "Light_conditions": "Darkness - lights lit",
        "Weather_conditions": "Raining",
        "Time_Period": "Night",
        "Casualty_class": "Passenger"
    },

    {
        "Day_of_week": "Sunday",
        "Area_accident_occured": "Rural village areas",
        "Lanes_or_Medians": "Two-way (divided with broken lines road marking)",
        "Types_of_Junction": "Crossing",
        "Road_surface_conditions": "Wet or damp",
        "Light_conditions": "Darkness - no lighting",
        "Weather_conditions": "Raining",
        "Time_Period": "Night",
        "Casualty_class": "Pedestrian"
    }
]

for i, case in enumerate(test_cases, start=1):
    risk, confidence = predict_risk(case)

    print(f"\n========== Test Case {i} ==========")
    print("Predicted Risk :", risk)
    print("Confidence :", round(confidence * 100, 2), "%")


========== Test Case 1 ==========
Predicted Risk : Low
Confidence : 47.89 %

========== Test Case 2 ==========
Predicted Risk : Medium
Confidence : 99.51 %

========== Test Case 3 ==========
Predicted Risk : High
Confidence : 100.0 %


In [5]:
user_data = {}

for column in feature_columns:
    value = input(f"Enter {column}: ")
    user_data[column] = value

risk, confidence = predict_risk(user_data)

print("\n========== Prediction ==========")
print("Predicted Risk :", risk)
print("Confidence :", round(confidence * 100, 2), "%")


========== Prediction ==========
Predicted Risk : Low
Confidence : 81.84 %


In [6]:
for column, categories in zip(feature_columns, preprocessor.named_transformers_['cat'].categories_):
    print(f"\n{column}")
    print(list(categories))


Day_of_week
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

Area_accident_occured
['  Market areas', '  Recreational areas', ' Church areas', ' Hospital areas', ' Industrial areas', ' Outside rural areas', 'Office areas', 'Other', 'Recreational areas', 'Residential areas', 'Rural village areas', 'Rural village areasOffice areas', 'School areas', 'Unknown']

Lanes_or_Medians
['Double carriageway (median)', 'One way', 'Two-way (divided with broken lines road marking)', 'Two-way (divided with solid lines road marking)', 'Undivided Two way', 'Unknown', 'other']

Types_of_Junction
['Crossing', 'No junction', 'O Shape', 'Other', 'T Shape', 'Unknown', 'X Shape', 'Y Shape']

Road_surface_conditions
['Dry', 'Flood over 3cm. deep', 'Snow', 'Wet or damp']

Light_conditions
['Darkness - lights lit', 'Darkness - lights unlit', 'Darkness - no lighting', 'Daylight']

Weather_conditions
['Cloudy', 'Fog or mist', 'Normal', 'Other', 'Raining', 'Raining and Windy', 'Snow'

In [7]:
user_data = {}

for column in feature_columns:
    value = input(f"Enter {column}: ")
    user_data[column] = value

risk, confidence = predict_risk(user_data)

print("\n========== Prediction ==========")
print("Predicted Risk :", risk)
print("Confidence :", round(confidence * 100, 2), "%")


========== Prediction ==========
Predicted Risk : High
Confidence : 91.92 %
